# DQN CarRacing-v3 — Google Colab Training

Make sure to set the runtime to **GPU**: Runtime → Change runtime type → T4 GPU.

Weights are saved to your Google Drive so they persist between sessions.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!apt-get install -y swig > /dev/null 2>&1
!pip install -q gymnasium[box2d] torch numpy

In [ ]:
# ── 2. Mount Google Drive (weights will be saved here) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Imports ───────────────────────────────────────────────────────────────
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
from collections import deque
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import clear_output

print(f"PyTorch: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

In [ ]:
# ── 4. Config ────────────────────────────────────────────────────────────────
# Weights are stored in Google Drive so they survive session restarts.
WEIGHTS_DIR = Path('/content/drive/MyDrive/dqn_carracing_weights')
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Epsilon-greedy
EPS_START   = 1.0
EPS_END     = 0.05
EPS_DECAY   = 200_000

# Training
TRAIN_START   = 5_000
TARGET_UPDATE = 1_000
SAVE_EVERY    = 10          # save checkpoint every N episodes
MAX_EPISODES  = 1_000
GAMMA         = 0.99
LR            = 1e-4
BATCH_SIZE    = 32
BUFFER_SIZE   = 50_000      # larger buffer — GPU has more RAM than CPU training
STACK_N       = 4
PRE_TRAINED   = True        # auto-resume from latest checkpoint if one exists

print(f"Weights dir: {WEIGHTS_DIR}")

In [ ]:
# ── 5. Model & helpers (src/model.py) ────────────────────────────────────────

def preprocess_without_graysscale(obs):
    """(96,96,3) uint8 → (3,84,96) uint8, drops bottom dashboard strip."""
    return obs[:84].transpose(2, 0, 1).copy()


class FrameStack:
    """Sliding window of the last STACK_N frames so the agent can perceive motion."""
    def __init__(self, n):
        self.n = n
        self.frames = deque(maxlen=n)

    def reset(self, frame):
        for _ in range(self.n):
            self.frames.append(frame)

    def push(self, frame):
        self.frames.append(frame)

    def get(self):
        return np.concatenate(self.frames, axis=0)  # (n*c, 84, 96)


class DQN(nn.Module):
    """Three conv layers + two FC layers; input is a stack of 4 RGB frames."""
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        conv_out = self._conv_out_size()
        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512), nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def _conv_out_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 12, 84, 96)
            return self.conv(dummy).view(1, -1).shape[1]

    def forward(self, x):
        x = self.conv(x).view(x.size(0), -1)
        return self.fc(x)


class ReplayBuffer:
    """Fixed-size circular replay buffer."""
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer   = []
        self.pos      = 0

    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.pos] = (state, action, reward, next_state, done)
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, device):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.uint8, device=device).float() / 255.0,
            torch.tensor(actions,               dtype=torch.int64,   device=device),
            torch.tensor(rewards,               dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.uint8, device=device).float() / 255.0,
            torch.tensor(dones,                 dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


print('Model classes ready.')

In [ ]:
# ── 6. Training helpers (src/train.py) ───────────────────────────────────────

def make_env():
    return gym.make('CarRacing-v3', continuous=False)


def select_action(state, eps, net, device):
    """Epsilon-greedy action selection."""
    if random.random() < eps:
        return random.randint(0, 4)
    with torch.no_grad():
        s = torch.tensor(state[None], dtype=torch.float32, device=device)
        return net(s).argmax(dim=1).item()


def compute_loss(batch, policy_net, target_net):
    """MSE loss between predicted Q-values and Bellman targets."""
    states, actions, rewards, next_states, dones = batch
    q_vals = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        max_next_q = target_net(next_states).max(1)[0]
        targets    = rewards + GAMMA * max_next_q * (1 - dones)
    return nn.functional.mse_loss(q_vals, targets)


print('Training helpers ready.')

In [ ]:
# ── 7. Live plot helper ───────────────────────────────────────────────────────

def plot_returns(returns, losses, episode):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(returns, alpha=0.4, label='Episode return')
    if len(returns) >= 10:
        smoothed = np.convolve(returns, np.ones(10)/10, mode='valid')
        ax1.plot(range(9, len(returns)), smoothed, label='10-ep avg')
    ax1.set_title(f'Return (ep {episode})')
    ax1.set_xlabel('Episode')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    if losses:
        ax2.plot(losses, alpha=0.5)
        ax2.set_title('Training loss (per update step)')
        ax2.set_xlabel('Update step')
        ax2.set_yscale('log')
        ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close(fig)


print('Plot helper ready.')

In [ ]:
# ── 8. Training loop ─────────────────────────────────────────────────────────
# Re-run this cell to continue training from the latest checkpoint.

env        = make_env()
policy_net = DQN(n_actions=5).to(device)
target_net = DQN(n_actions=5).to(device)

# Resume from the latest checkpoint if PRE_TRAINED is True
start_episode = 1
checkpoints   = sorted(glob.glob(str(WEIGHTS_DIR / 'dqn_ep*.pt')), key=os.path.getmtime)
if checkpoints and PRE_TRAINED:
    latest = checkpoints[-1]
    policy_net.load_state_dict(torch.load(latest, map_location=device))
    try:
        start_episode = int(Path(latest).stem.replace('dqn_ep', '')) + 1
    except ValueError:
        pass
    print(f'Resumed from {latest}  (starting at ep {start_episode})')
else:
    print('Starting fresh training.')

target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer   = torch.optim.Adam(policy_net.parameters(), lr=LR)
buffer      = ReplayBuffer(BUFFER_SIZE)
frame_stack = FrameStack(STACK_N)

all_returns = []
all_losses  = []
total_steps = (start_episode - 1) * 1000  # rough step estimate for epsilon schedule

for episode in range(start_episode, MAX_EPISODES + 1):
    obs, _  = env.reset()
    frame_stack.reset(preprocess_without_graysscale(obs))
    state   = frame_stack.get()
    ep_ret  = 0.0
    done    = False

    while not done:
        eps    = max(EPS_END, EPS_START - (EPS_START - EPS_END) * total_steps / EPS_DECAY)
        action = select_action(state, eps, policy_net, device)

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        frame_stack.push(preprocess_without_graysscale(next_obs))
        next_state = frame_stack.get()
        buffer.push(state, action, reward, next_state, float(done))
        state      = next_state
        ep_ret    += reward
        total_steps += 1

        if len(buffer) >= TRAIN_START:
            batch = buffer.sample(BATCH_SIZE, device)
            loss  = compute_loss(batch, policy_net, target_net)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy_net.parameters(), 10)
            optimizer.step()
            all_losses.append(loss.item())

        if total_steps % TARGET_UPDATE == 0:
            target_net.load_state_dict(policy_net.state_dict())

    all_returns.append(ep_ret)
    print(f'Ep {episode:4d} | Return {ep_ret:8.1f} | Eps {eps:.3f} | Steps {total_steps}')

    if episode % SAVE_EVERY == 0:
        ckpt = str(WEIGHTS_DIR / f'dqn_ep{episode}.pt')
        torch.save(policy_net.state_dict(), ckpt)
        print(f'  -> Saved {ckpt}')
        plot_returns(all_returns, all_losses, episode)

torch.save(policy_net.state_dict(), str(WEIGHTS_DIR / 'dqn_final.pt'))
env.close()
print('Training complete.')

In [ ]:
# ── 9. Evaluate a checkpoint (src/evaluate.py) ───────────────────────────────
# Change MODEL_PATH to any checkpoint you want to evaluate.

MODEL_PATH  = str(WEIGHTS_DIR / 'dqn_final.pt')   # or e.g. 'dqn_ep100.pt'
N_EPISODES  = 5

def run_episode(env, net, device):
    obs, _      = env.reset()
    fs          = FrameStack(STACK_N)
    fs.reset(preprocess_without_graysscale(obs))
    total_ret   = 0.0
    done        = False
    while not done:
        state = fs.get()
        with torch.no_grad():
            s      = torch.tensor(state[None], dtype=torch.float32, device=device)
            action = net(s).argmax(dim=1).item()
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        fs.push(preprocess_without_graysscale(obs))
        total_ret += reward
    return total_ret


eval_env = gym.make('CarRacing-v3', continuous=False)
eval_net = DQN(n_actions=5).to(device)
eval_net.load_state_dict(torch.load(MODEL_PATH, map_location=device))
eval_net.eval()

returns = []
for i in range(N_EPISODES):
    r = run_episode(eval_env, eval_net, device)
    returns.append(r)
    print(f'Episode {i+1}: {r:.1f}')

print(f'\nMean return over {N_EPISODES} episodes: {np.mean(returns):.1f}')
eval_env.close()

In [ ]:
# ── 10. Download a checkpoint to your machine ─────────────────────────────────
# Alternatively, the weights are already in your Google Drive.

from google.colab import files

DOWNLOAD_CKPT = str(WEIGHTS_DIR / 'dqn_final.pt')   # change as needed
files.download(DOWNLOAD_CKPT)